# Onboard Radiology Revenue Cycle Tables to DQ Monitoring

**Purpose**: Registers 11 tables from `serverless_stable_jc9zgx_catalog.radiology_revenue_cycle` into the Data Quality monitoring framework.

**What this notebook does**:
1. Creates a new "Finance" data domain (domain_id=7)
2. Registers catalog/schema entries for the radiology data
3. Registers 11 tables across Finance and Revenue Cycle domains
4. Registers columns (including placeholder columns for cross-table checks)
5. Creates 8 DQ rules from the "Two Reports, One Truth" demo
6. Creates column-rule assignments
7. Generates 30 days of synthetic trend data showing gradual quality degradation

**Source**: `serverless_stable_jc9zgx_catalog.radiology_revenue_cycle`  
**Target**: `serverless_stable_jc9zgx_catalog.data_quality`

> ⚠️ **Note**: This notebook is not idempotent. Running it twice will create duplicate records. Delete inserted rows first if re-running.

In [0]:
%sql
-- Explore catalog table structure to understand domain linkage
SELECT * FROM serverless_stable_jc9zgx_catalog.data_quality.catalog

In [0]:
%sql
-- Create Finance domain (domain_id=7)
-- Thresholds aligned with Revenue Cycle domain (min=75, warning=85, critical=60)
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.data_domain
VALUES (
  7,                    -- domain_id
  1,                    -- organization_id
  2,                    -- business_owner_id (same owner as rev_cycle_catalog)
  'Finance',            -- domain_name
  'Financial reporting tables including claims, payments, and payer contracts for Pinnacle Radiology Partners',
  75.0,                 -- dq_threshold_min
  85.0,                 -- dq_threshold_warning
  60.0,                 -- dq_threshold_critical
  false,                -- inherits_org_threshold
  CURRENT_TIMESTAMP()   -- created_at
)

In [0]:
%sql
-- Create catalog and schema entries
-- Finance catalog (catalog_id=7, domain_id=7)
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.catalog
VALUES (7, 7, 2, 'finance_catalog', 'Databricks', 'prod', 75.0, 85.0, 60.0, true, CURRENT_TIMESTAMP());

-- Schema for Finance tables (claims, payments, payers)
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.`schema`
VALUES (20, 7, 2, 'radiology_finance', CURRENT_TIMESTAMP());

-- Schema for Revenue Cycle tables (encounters, denials, AR, auths)
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.`schema`
VALUES (21, 2, 2, 'radiology_revenue_cycle', CURRENT_TIMESTAMP())

In [0]:
%sql
-- Register 11 RCM tables in dq_table
-- Finance domain (schema_id=20): claim_header, claim_line, payment, payer
-- Revenue Cycle domain (schema_id=21): encounter, denial, ar_aging_snapshot, authorization, patient, provider, service_location

INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_table VALUES
  -- Finance domain tables
  (98,  20, 2, 'claim_header',      'Claims filed to payers for radiology services', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  (99,  20, 2, 'claim_line',        'Individual procedure lines within each claim', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  (100, 20, 2, 'payment',           'Payments received from payers and patients', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  (101, 20, 2, 'payer',             'Insurance payer reference table with canonical names', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  -- Revenue Cycle domain tables
  (102, 21, 2, 'encounter',         'Patient encounters/studies at imaging locations', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  (103, 21, 2, 'denial',            'Claim denials with CARC/RARC codes', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  (104, 21, 2, 'ar_aging_snapshot', 'Daily AR aging buckets by location and payer', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  (105, 21, 2, 'authorization',     'Prior authorization records for advanced imaging', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  (106, 21, 2, 'patient',           'Patient demographics', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  (107, 21, 2, 'provider',          'Radiologist and referring physician directory', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()),
  (108, 21, 2, 'service_location',  'Imaging center and hospital-based locations', true, 'daily', CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP())

In [0]:
%sql
-- Register columns in dq_column (starting at column_id=486)
-- Includes physical columns for each table PLUS 3 placeholder columns for cross-table checks
-- Placeholder columns use data_type='custom_cross_table' to indicate they're not physical columns

-- claim_header (table 98): columns 486-497
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(486, 98, 'claim_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(487, 98, 'encounter_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(488, 98, 'patient_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(489, 98, 'payer_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(490, 98, 'payer_name', 'string', true, false, false, CURRENT_TIMESTAMP()),
(491, 98, 'claim_status', 'string', false, false, false, CURRENT_TIMESTAMP()),
(492, 98, 'total_charge_amount', 'decimal', true, false, false, CURRENT_TIMESTAMP()),
(493, 98, 'total_allowed_amount', 'decimal', true, false, false, CURRENT_TIMESTAMP()),
(494, 98, 'total_paid_amount', 'decimal', true, false, false, CURRENT_TIMESTAMP()),
(495, 98, 'is_rebill', 'boolean', true, false, false, CURRENT_TIMESTAMP()),
(496, 98, 'original_claim_id', 'int', true, false, false, CURRENT_TIMESTAMP()),
(497, 98, 'Cross-Table: Rebill Duplicate Check', 'custom_cross_table', false, false, false, CURRENT_TIMESTAMP());

-- claim_line (table 99): columns 498-507
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(498, 99, 'claim_line_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(499, 99, 'claim_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(500, 99, 'cpt_code', 'string', false, false, false, CURRENT_TIMESTAMP()),
(501, 99, 'modifier_1', 'string', true, false, false, CURRENT_TIMESTAMP()),
(502, 99, 'charge_amount', 'decimal', false, false, false, CURRENT_TIMESTAMP()),
(503, 99, 'allowed_amount', 'decimal', true, false, false, CURRENT_TIMESTAMP()),
(504, 99, 'paid_amount', 'decimal', true, false, false, CURRENT_TIMESTAMP()),
(505, 99, 'adjustment_reason_code', 'string', true, false, false, CURRENT_TIMESTAMP()),
(506, 99, 'Cross-Table: TC/26 Component Overlap', 'custom_cross_table', false, false, false, CURRENT_TIMESTAMP()),
(507, 99, 'Cross-Table: Allowed vs Payment Reconciliation', 'custom_cross_table', false, false, false, CURRENT_TIMESTAMP());

-- payment (table 100): columns 508-513
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(508, 100, 'payment_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(509, 100, 'claim_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(510, 100, 'claim_line_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(511, 100, 'payer_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(512, 100, 'payment_amount', 'decimal', false, false, false, CURRENT_TIMESTAMP()),
(513, 100, 'payment_date', 'date', false, false, false, CURRENT_TIMESTAMP());

-- payer (table 101): columns 514-516
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(514, 101, 'payer_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(515, 101, 'payer_name', 'string', false, false, false, CURRENT_TIMESTAMP()),
(516, 101, 'payer_category', 'string', false, false, false, CURRENT_TIMESTAMP());

-- encounter (table 102): columns 517-522
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(517, 102, 'encounter_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(518, 102, 'patient_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(519, 102, 'provider_npi', 'string', false, false, false, CURRENT_TIMESTAMP()),
(520, 102, 'location_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(521, 102, 'study_date', 'date', false, false, false, CURRENT_TIMESTAMP()),
(522, 102, 'modality', 'string', false, false, false, CURRENT_TIMESTAMP());

-- denial (table 103): columns 523-527
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(523, 103, 'denial_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(524, 103, 'claim_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(525, 103, 'carc_code', 'string', false, false, false, CURRENT_TIMESTAMP()),
(526, 103, 'denial_date', 'date', false, false, false, CURRENT_TIMESTAMP()),
(527, 103, 'denied_amount', 'decimal', false, false, false, CURRENT_TIMESTAMP());

-- ar_aging_snapshot (table 104): columns 528-533
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(528, 104, 'snapshot_date', 'date', false, true, false, CURRENT_TIMESTAMP()),
(529, 104, 'payer_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(530, 104, 'location_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(531, 104, 'aging_bucket', 'string', false, false, false, CURRENT_TIMESTAMP()),
(532, 104, 'outstanding_amount', 'decimal', false, false, false, CURRENT_TIMESTAMP()),
(533, 104, 'snapshot_source', 'string', false, false, false, CURRENT_TIMESTAMP());

-- authorization (table 105): columns 534-538
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(534, 105, 'auth_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(535, 105, 'patient_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(536, 105, 'payer_id', 'int', false, false, true, CURRENT_TIMESTAMP()),
(537, 105, 'cpt_code', 'string', false, false, false, CURRENT_TIMESTAMP()),
(538, 105, 'auth_status', 'string', false, false, false, CURRENT_TIMESTAMP());

-- patient (table 106): columns 539-542
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(539, 106, 'patient_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(540, 106, 'mrn', 'string', false, false, false, CURRENT_TIMESTAMP()),
(541, 106, 'dob', 'date', false, false, false, CURRENT_TIMESTAMP()),
(542, 106, 'sex', 'string', true, false, false, CURRENT_TIMESTAMP());

-- provider (table 107): columns 543-545
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(543, 107, 'provider_npi', 'string', false, true, false, CURRENT_TIMESTAMP()),
(544, 107, 'provider_name', 'string', false, false, false, CURRENT_TIMESTAMP()),
(545, 107, 'specialty', 'string', true, false, false, CURRENT_TIMESTAMP());

-- service_location (table 108): columns 546-548
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_column VALUES
(546, 108, 'location_id', 'int', false, true, false, CURRENT_TIMESTAMP()),
(547, 108, 'location_name', 'string', false, false, false, CURRENT_TIMESTAMP()),
(548, 108, 'location_type', 'string', false, false, false, CURRENT_TIMESTAMP())

In [0]:
%sql
-- Create 8 DQ rules for the radiology revenue cycle checks
-- Rules 27-34 cover all 6 DQ dimensions from the "Two Reports, One Truth" demo
-- Cross-table rules use rule_type='custom_cross_table'

INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_rule VALUES
-- Rule 27: Completeness - NULL allowed_amount on paid claims
(27, 1, 1, NULL, 'paid_claim_line_allowed_not_null',
 'Paid claims must have allowed_amount populated. NULL indicates ERA/835 remittance was not matched to the claim.',
 'null_check', 2.0,
 'SELECT COUNT(*) AS total_paid_lines, SUM(CASE WHEN cl.allowed_amount IS NULL THEN 1 ELSE 0 END) AS failing_rows, ROUND(100.0 * SUM(CASE WHEN cl.allowed_amount IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS failure_pct FROM serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_line cl JOIN serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_header ch ON cl.claim_id = ch.claim_id WHERE ch.claim_status = ''paid''',
 'active', 1, false, 'rcm_onboarding', CURRENT_TIMESTAMP(), NULL),

-- Rule 28: Validity - Invalid CARC codes
(28, 3, 7, NULL, 'denial_carc_code_valid',
 'All CARC codes on denials must exist in the X12 835 standard code set. Invalid codes indicate clearinghouse translation errors.',
 'referential_check', 1.5,
 'WITH valid_carcs AS (SELECT explode(array(''CO-4'',''CO-16'',''CO-45'',''CO-50'',''CO-197'',''PR-1'',''PR-2'',''PR-96'',''OA-23'',''CO-42'')) AS valid_code) SELECT COUNT(*) AS total_denials, SUM(CASE WHEN vc.valid_code IS NULL THEN 1 ELSE 0 END) AS invalid_carc_count FROM serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.denial d LEFT JOIN valid_carcs vc ON d.carc_code = vc.valid_code',
 'active', 1, false, 'rcm_onboarding', CURRENT_TIMESTAMP(), NULL),

-- Rule 29: Consistency - Payer name mismatches
(29, 4, 7, NULL, 'claim_payer_name_matches_reference',
 'claim_header.payer_name should match canonical payer.payer_name for the same payer_id. Mismatches indicate naming inconsistencies from multiple entry points.',
 'referential_check', 1.0,
 'SELECT COUNT(*) AS total_claims, SUM(CASE WHEN ch.payer_name != p.payer_name THEN 1 ELSE 0 END) AS mismatched_names FROM serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_header ch JOIN serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.payer p ON ch.payer_id = p.payer_id',
 'active', 1, false, 'rcm_onboarding', CURRENT_TIMESTAMP(), NULL),

-- Rule 30: Uniqueness - Rebill duplicates (cross-table)
(30, 5, 3, NULL, 'claim_no_duplicate_service',
 'Cross-table check: No duplicate claims for same service (patient_id + service_date + cpt_code + charge_amount). Unflagged rebills before Oct 2025 create phantom revenue.',
 'custom_cross_table', 3.0,
 'SELECT COUNT(*) AS duplicate_groups, SUM(duplicate_count - 1) AS excess_claims FROM (SELECT ch.patient_id, cl.service_date, cl.cpt_code, cl.charge_amount, COUNT(DISTINCT ch.claim_id) AS duplicate_count FROM serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_header ch JOIN serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_line cl ON ch.claim_id = cl.claim_id WHERE ch.claim_status NOT IN (''void'') GROUP BY ch.patient_id, cl.service_date, cl.cpt_code, cl.charge_amount HAVING COUNT(DISTINCT ch.claim_id) > 1)',
 'active', 1, false, 'rcm_onboarding', CURRENT_TIMESTAMP(), NULL),

-- Rule 31: Timeliness - Stale AR snapshots
(31, 6, 8, NULL, 'ar_snapshot_within_24h',
 'AR snapshots must be refreshed within 24 hours. Stale feeds from hospital systems cause Days in AR to diverge between reports.',
 'freshness_check', 2.0,
 'SELECT snapshot_source, MAX(snapshot_date) AS latest_snapshot, DATEDIFF(CURRENT_DATE(), MAX(snapshot_date)) AS days_stale FROM serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.ar_aging_snapshot GROUP BY snapshot_source',
 'active', 1, false, 'rcm_onboarding', CURRENT_TIMESTAMP(), NULL),

-- Rule 32: Accuracy A - TC/26 component overlap (cross-table)
(32, 2, 4, NULL, 'claim_line_no_component_overlap',
 'Cross-table check: Same CPT should not appear as both global (no modifier) and component (TC/26) on same claim. This triple-counts revenue.',
 'custom_cross_table', 3.0,
 'SELECT COUNT(DISTINCT cl_global.claim_id) AS claims_with_overlap FROM serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_line cl_global JOIN serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_line cl_component ON cl_global.claim_id = cl_component.claim_id AND cl_global.cpt_code = cl_component.cpt_code AND cl_global.claim_line_id != cl_component.claim_line_id WHERE cl_global.modifier_1 IS NULL AND cl_component.modifier_1 IN (''26'', ''TC'')',
 'active', 1, false, 'rcm_onboarding', CURRENT_TIMESTAMP(), NULL),

-- Rule 33: Accuracy B - allowed = charge (clearinghouse error)
(33, 2, 4, NULL, 'allowed_not_equal_to_billed',
 'Contracted payers should never have allowed_amount = charge_amount. Indicates clearinghouse mapping error where charge was incorrectly placed in allowed field.',
 'range_check', 2.0,
 'SELECT COUNT(*) AS total_lines, SUM(CASE WHEN cl.allowed_amount = cl.charge_amount THEN 1 ELSE 0 END) AS allowed_equals_billed FROM serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_line cl JOIN serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_header ch ON cl.claim_id = ch.claim_id WHERE cl.allowed_amount IS NOT NULL',
 'active', 1, false, 'rcm_onboarding', CURRENT_TIMESTAMP(), NULL),

-- Rule 34: Accuracy C - Payment exceeds allowed (cross-table)
(34, 2, 4, NULL, 'payment_not_exceeding_allowed',
 'Cross-table check: Payments should not systematically exceed allowed amounts (>5% tolerance). Indicates stale fee schedules where allowed was not updated after contract renegotiation.',
 'custom_cross_table', 1.5,
 'SELECT COUNT(*) AS lines_checked, SUM(CASE WHEN pmt.payment_amount > cl.allowed_amount * 1.05 THEN 1 ELSE 0 END) AS payment_exceeds_allowed FROM serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_line cl JOIN serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.claim_header ch ON cl.claim_id = ch.claim_id JOIN serverless_stable_jc9zgx_catalog.radiology_revenue_cycle.payment pmt ON cl.claim_line_id = pmt.claim_line_id WHERE cl.allowed_amount IS NOT NULL AND cl.allowed_amount > 0 AND ch.filing_date BETWEEN ''2026-01-01'' AND ''2026-02-28''',
 'active', 1, false, 'rcm_onboarding', CURRENT_TIMESTAMP(), NULL)

In [0]:
%sql
-- Create column_rule_assignments (1456-1463)
-- Maps each rule to its target column (including placeholders for cross-table rules)
-- Column ID reference:
--   503 = claim_line.allowed_amount
--   525 = denial.carc_code
--   490 = claim_header.payer_name
--   497 = Cross-Table: Rebill Duplicate Check (placeholder on claim_header)
--   528 = ar_aging_snapshot.snapshot_date
--   506 = Cross-Table: TC/26 Component Overlap (placeholder on claim_line)
--   507 = Cross-Table: Allowed vs Payment Reconciliation (placeholder on claim_line)

INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.column_rule_assignment VALUES
(1456, 503, 27, 0.05, 'lt', NULL, true, CURRENT_TIMESTAMP()),
(1457, 525, 28, 0.01, 'lt', NULL, true, CURRENT_TIMESTAMP()),
(1458, 490, 29, 0.05, 'lt', NULL, true, CURRENT_TIMESTAMP()),
(1459, 497, 30, 0.01, 'lt', NULL, true, CURRENT_TIMESTAMP()),
(1460, 528, 31, 1.0,  'lt', NULL, true, CURRENT_TIMESTAMP()),
(1461, 506, 32, 0.01, 'lt', NULL, true, CURRENT_TIMESTAMP()),
(1462, 503, 33, 0.01, 'lt', NULL, true, CURRENT_TIMESTAMP()),
(1463, 507, 34, 0.05, 'lt', NULL, true, CURRENT_TIMESTAMP())

In [0]:
%sql
-- Generate 30 daily dq_run entries (run_id 32-61)
-- Daily from 2026-04-18 to 2026-05-17
INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.dq_run
SELECT
  run_id,
  CAST(DATE_ADD('2026-04-17', run_id - 31) AS TIMESTAMP) AS run_timestamp,
  'scheduled' AS triggered_by,
  NULL AS pipeline_job_id,
  'full' AS run_scope,
  'completed' AS status
FROM (SELECT explode(sequence(32, 61)) AS run_id)

In [0]:
%sql
-- Generate 240 column_rule_result rows (30 days × 8 rules)
-- Degradation curve: days 1-10 near clean (0.5%), days 11-30 linear ramp to final bad values
-- Scoring formula:
--   raw_score = 100 * (1 - failure_rate)
--   weighted_score = MAX(0, 100 - (failure_rate * 100 * severity_weight))
--   passed = weighted_score >= 75

INSERT INTO serverless_stable_jc9zgx_catalog.data_quality.column_rule_result
WITH rule_params AS (
  SELECT * FROM (VALUES
    (1456, 0.08,  2.0, 95544),   -- Completeness: 8% NULL allowed
    (1457, 0.256, 1.5, 5760),    -- Validity: 25.6% invalid CARC
    (1458, 1.0,   1.0, 37541),   -- Consistency: 100% payer name mismatch
    (1459, 0.03,  3.0, 37541),   -- Uniqueness: 3% rebill duplicates
    (1460, 0.75,  2.0, 5190),    -- Timeliness: 75% stale snapshots
    (1461, 0.015, 3.0, 95544),   -- Accuracy A: 1.5% TC/26 overlap
    (1462, 1.0,   2.0, 95544),   -- Accuracy B: 100% allowed=charge
    (1463, 0.12,  1.5, 95544)    -- Accuracy C: 12% stale fee schedule
  ) AS t(assignment_id, final_failure_rate, severity_weight, total_records)
),
days AS (
  SELECT explode(sequence(1, 30)) AS day_num
),
computed AS (
  SELECT
    rp.assignment_id,
    d.day_num,
    d.day_num + 31 AS run_id,
    -- Degradation: flat at 0.5% for days 1-10, linear ramp to final for days 11-30
    0.005 + (rp.final_failure_rate - 0.005) * GREATEST(0.0, CAST(d.day_num - 10 AS DOUBLE) / 20.0) AS day_failure_rate,
    rp.severity_weight,
    rp.total_records
  FROM rule_params rp
  CROSS JOIN days d
),
scored AS (
  SELECT
    c.*,
    ROUND(100.0 * (1.0 - c.day_failure_rate), 1) AS raw_score,
    GREATEST(0.0, ROUND(100.0 - (c.day_failure_rate * 100.0 * c.severity_weight), 1)) AS weighted_score,
    CAST(c.total_records * c.day_failure_rate AS BIGINT) AS failed_records,
    CAST(DATE_ADD('2026-04-17', c.run_id - 31) AS TIMESTAMP) AS evaluated_at
  FROM computed c
)
SELECT
  45105 + ROW_NUMBER() OVER (ORDER BY s.run_id, s.assignment_id) AS result_id,
  s.run_id,
  s.assignment_id,
  CASE WHEN s.weighted_score >= 75.0 THEN true ELSE false END AS passed,
  CAST(s.total_records AS BIGINT) AS total_records,
  s.failed_records,
  s.day_failure_rate AS failure_rate,
  s.raw_score,
  s.weighted_score,
  false AS egregious_flag,
  NULL AS executed_rule_sql,
  NULL AS result_detail,
  s.evaluated_at
FROM scored s

In [0]:
%sql
-- Verification: Count records added to each metadata table
SELECT 'data_domain' AS table_name, COUNT(*) AS total, SUM(CASE WHEN domain_id = 7 THEN 1 ELSE 0 END) AS new_rows FROM serverless_stable_jc9zgx_catalog.data_quality.data_domain
UNION ALL SELECT 'catalog', COUNT(*), SUM(CASE WHEN catalog_id = 7 THEN 1 ELSE 0 END) FROM serverless_stable_jc9zgx_catalog.data_quality.catalog
UNION ALL SELECT 'schema', COUNT(*), SUM(CASE WHEN schema_id >= 20 THEN 1 ELSE 0 END) FROM serverless_stable_jc9zgx_catalog.data_quality.`schema`
UNION ALL SELECT 'dq_table', COUNT(*), SUM(CASE WHEN table_id >= 98 THEN 1 ELSE 0 END) FROM serverless_stable_jc9zgx_catalog.data_quality.dq_table
UNION ALL SELECT 'dq_column', COUNT(*), SUM(CASE WHEN column_id >= 486 THEN 1 ELSE 0 END) FROM serverless_stable_jc9zgx_catalog.data_quality.dq_column
UNION ALL SELECT 'dq_rule', COUNT(*), SUM(CASE WHEN rule_id >= 27 THEN 1 ELSE 0 END) FROM serverless_stable_jc9zgx_catalog.data_quality.dq_rule
UNION ALL SELECT 'column_rule_assignment', COUNT(*), SUM(CASE WHEN assignment_id >= 1456 THEN 1 ELSE 0 END) FROM serverless_stable_jc9zgx_catalog.data_quality.column_rule_assignment
UNION ALL SELECT 'dq_run', COUNT(*), SUM(CASE WHEN run_id >= 32 THEN 1 ELSE 0 END) FROM serverless_stable_jc9zgx_catalog.data_quality.dq_run
UNION ALL SELECT 'column_rule_result', COUNT(*), SUM(CASE WHEN result_id >= 45106 THEN 1 ELSE 0 END) FROM serverless_stable_jc9zgx_catalog.data_quality.column_rule_result

In [0]:
%sql
-- Show latest run scores (day 30 = run_id 61) ordered by worst first
SELECT 
  r.rule_name,
  qd.dimension_name,
  crr.failure_rate,
  crr.raw_score,
  crr.weighted_score,
  crr.passed,
  crr.total_records,
  crr.failed_records
FROM serverless_stable_jc9zgx_catalog.data_quality.column_rule_result crr
JOIN serverless_stable_jc9zgx_catalog.data_quality.column_rule_assignment cra ON crr.assignment_id = cra.assignment_id
JOIN serverless_stable_jc9zgx_catalog.data_quality.dq_rule r ON cra.rule_id = r.rule_id
JOIN serverless_stable_jc9zgx_catalog.data_quality.quality_dimension qd ON r.dimension_id = qd.dimension_id
WHERE crr.run_id = 61
ORDER BY crr.weighted_score ASC

In [0]:
%sql
-- Verify trend degradation: day 1 vs day 30 scores
SELECT 
  r.rule_name,
  early.weighted_score AS day1_score,
  late.weighted_score AS day30_score,
  ROUND(late.weighted_score - early.weighted_score, 1) AS score_change
FROM serverless_stable_jc9zgx_catalog.data_quality.column_rule_result early
JOIN serverless_stable_jc9zgx_catalog.data_quality.column_rule_result late 
  ON early.assignment_id = late.assignment_id
JOIN serverless_stable_jc9zgx_catalog.data_quality.column_rule_assignment cra 
  ON early.assignment_id = cra.assignment_id
JOIN serverless_stable_jc9zgx_catalog.data_quality.dq_rule r 
  ON cra.rule_id = r.rule_id
WHERE early.run_id = 32 AND late.run_id = 61
ORDER BY score_change ASC